## 1. Setup and Environment

Load shared config and suppress non-critical warnings.

In [1]:
import sys
sys.path.append(r"D:\Quant\Multi_factor_project")
from config import *

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Environment loaded successfully")

环境加载完成


## 2. Load Cleaned/Raw Inputs

Load the daily price panel (from Chapter 01), the raw monthly constituent list, and the CSI 300 benchmark index series.

In [2]:
# Daily price panel
daily_panel = pd.read_parquet(f"{DATA_CLEAN}/daily_panel.parquet")
daily_panel['trade_date'] = pd.to_datetime(daily_panel['trade_date'])

# Monthly constituent list
universe_raw = pd.read_parquet(f"{DATA_RAW}/industry/hs300_monthly_universe.parquet")

# CSI 300 index
hs300 = pd.read_parquet(f"{DATA_RAW}/daily/hs300_index.parquet")
hs300['trade_date'] = pd.to_datetime(hs300['trade_date'])

print("daily_panel:", daily_panel.shape)
print("universe_raw:", universe_raw.shape)
print("hs300:", hs300.shape)

daily_panel: (1381265, 9)
universe_raw: (36000, 5)
hs300: (2431, 11)


## 3. Extract Monthly Constituent List

Reduce the raw constituent table to a `(month, ts_code)` lookup, and sanity-check that each month has ~300 constituents.

In [4]:
monthly_universe = universe_raw[['month', 'con_code']].copy()
print(monthly_universe.shape)                       # should be approx. (36000, 2)
print(monthly_universe.head())
print(monthly_universe.groupby('month').size().describe())  # approx. 300 per month

(36000, 2)
     month   con_code
0  2015-01  000001.SZ
1  2015-01  000002.SZ
2  2015-01  000009.SZ
3  2015-01  000024.SZ
4  2015-01  000027.SZ
count    120.0
mean     300.0
std        0.0
min      300.0
25%      300.0
50%      300.0
75%      300.0
max      300.0
dtype: float64


## 4. Compute Monthly Returns

Apply the adjustment factor to raw close prices to obtain post-rights-adjusted prices, take the month-end price per stock, and compute month-over-month returns.

In [5]:
def calc_monthly_returns(daily_panel):
    """Compute monthly returns using post-rights-adjusted (hfq-style) prices."""
    # Load and merge the adjustment factors
    adj_dir = f"{DATA_RAW}/adj_factor"
    adj = pd.concat(
        [pd.read_parquet(f"{adj_dir}/{f}") for f in os.listdir(adj_dir)],
        ignore_index=True
    )
    adj['trade_date'] = pd.to_datetime(adj['trade_date'])
    
    df = daily_panel.merge(
        adj[['ts_code', 'trade_date', 'adj_factor']],
        on=['ts_code', 'trade_date'], how='left'
    )
    df['adj_close'] = df['close'] * df['adj_factor']  # post-rights-adjusted price
    df['month'] = df['trade_date'].dt.to_period('M')
    
    monthly_close = (df.sort_values('trade_date')
                       .groupby(['ts_code', 'month'])['adj_close']
                       .last()
                       .reset_index())
    
    monthly_close = monthly_close.sort_values(['ts_code', 'month'])
    monthly_close['ret_1m'] = (monthly_close
                                .groupby('ts_code')['adj_close']
                                .pct_change(1))
    monthly_close = monthly_close.dropna(subset=['ret_1m'])
    
    print(f"Monthly return calculation complete: {monthly_close.shape}")
    return monthly_close

monthly_ret = calc_monthly_returns(daily_panel)

月度收益率计算完成：(68287, 4)


## 5. Build the Base Monthly Panel

Inner-join monthly returns with the constituent list so that only stock-months where the stock was actually an HS300 constituent are retained. This is the point-in-time-correct research universe used for factor construction.

In [6]:
def build_base_panel(monthly_ret, monthly_universe):
    """
    Merge monthly returns with the constituent list
    to produce the base monthly panel.
    """
    # Flag whether the stock was an HS300 constituent in that month
    panel = monthly_ret.merge(
        monthly_universe.rename(columns={'con_code': 'ts_code'}),
        on=['ts_code', 'month'],
        how='inner'  # keep only records within the constituent universe
    )
    
    print(f"Base panel shape: {panel.shape}")
    print(f"\nConstituent count per month (last 6 months):")
    print(panel.groupby('month')['ts_code'].count().tail(6))
    print(f"\nColumns: {list(panel.columns)}")
    
    return panel

base_panel = build_base_panel(monthly_ret, monthly_universe)

基础面板形状：(35255, 4)

每月成分股数量（后6个月）：
month
2024-07    300
2024-08    300
2024-09    300
2024-10    300
2024-11    300
2024-12    300
Freq: M, Name: ts_code, dtype: int64

列名：['ts_code', 'month', 'adj_close', 'ret_1m']


## 6. Save Base Panel

Persist the base monthly panel to the clean data folder for use in factor construction (Chapter 03).

In [7]:
base_panel.to_parquet(f"{DATA_CLEAN}/base_panel.parquet")
print("OK base_panel.parquet saved")
print(f"Shape: {base_panel.shape}")
base_panel.head(3)

✓ base_panel.parquet 保存完成
形状：(35255, 4)


,ts_code,month,adj_close,ret_1m
0,000001.SZ,2015-02,994.04546,0.004307
1,000001.SZ,2015-03,1119.10050,0.125804
2,000001.SZ,2015-04,1436.09980,0.283263
